In [1]:
import pandas as pd
import re
from collections import Counter

# 엑셀 불러오기
df = pd.read_excel('./data/DB연동_공고데이터.xlsx')

# TECH_STACK 문자열 처리
stack_series = df["TECH_STACK"].fillna("").astype(str)

# 기술스택 분리
raw_tokens = []

for text in stack_series:
    parts = re.split(r"\s*,\s*", text.strip())   # 쉼표 기준 분리
    for p in parts:
        p = p.strip()
        if p:
            raw_tokens.append(p)

# 중복 제거한 원본 표현 목록
raw_unique_stacks = sorted(set(raw_tokens))

# 데이터프레임으로 만들기
raw_stack_df = pd.DataFrame({"raw_stack": raw_unique_stacks})

print(raw_stack_df)
print("원본 표현 개수:", len(raw_unique_stacks))

          raw_stack
0              .Net
1    .Net Framrwork
2          2종보통운전면허
3           ADsP 자격
4                AI
..              ...
358            패턴인식
359             풀스택
360          프레젠테이션
361             플래시
362              한컴

[363 rows x 1 columns]
원본 표현 개수: 363


In [2]:
counter = Counter(raw_tokens)

freq_df = pd.DataFrame(counter.items(), columns=["raw_stack", "count"])
freq_df = freq_df.sort_values(["count", "raw_stack"], ascending=[False, True]).reset_index(drop=True)

print(freq_df.head(100))

       raw_stack  count
0           JAVA    300
1         Python    259
2     Javascript    171
3          React    148
4            CSS    138
..           ...    ...
95         Kafka      4
96           RNN      4
97   Restful API      4
98         Swift      4
99  Temporal CNN      4

[100 rows x 2 columns]


In [3]:
raw_unique_stacks

['.Net',
 '.Net Framrwork',
 '2종보통운전면허',
 'ADsP 자격',
 'AI',
 'AI Agent',
 'AJAX',
 'AL/ML 도메인',
 'API',
 'API Git',
 'ASP',
 'ASP.NET',
 'AWS',
 'Adobe XD',
 'Agent',
 'Ai',
 'Ajax',
 'Android',
 'Android Studio',
 'Angular',
 'Apache',
 'Apache Spark',
 'Arduino',
 'Asana',
 'Azure',
 'Boot',
 'BootStrap',
 'Bootstrap',
 'C',
 'C#',
 'C++',
 'C++. Python',
 'C/C++',
 'CAD',
 'CCIE',
 'CCNA',
 'CCNP',
 'CISCO. Docker',
 'CNN',
 'CSS',
 'CSS 3',
 'CSS3',
 'CUDA',
 'Cafe24',
 'Caffe',
 'Chart.js',
 'ChatGPT',
 'ChatGPT API',
 'Claude',
 'Cloud',
 'C언어',
 'DB',
 'DBMS',
 'DL',
 'Dart',
 'Database ORM',
 'DeepLearning',
 'Diango',
 'Discord',
 'Django',
 'Docker',
 'Docker Compose',
 'ELK',
 'ERP',
 'ESP32',
 'ETL',
 'Eclipse',
 'Excel',
 'Excle',
 'FastAPI',
 'FastAPI(Python)',
 'Figma',
 'Flask',
 'Flow',
 'Flutter',
 'Fullter',
 'GA',
 'GCP',
 'GIT',
 'GO Java',
 'GPIO',
 'GPT',
 'GPT API',
 'GSAP',
 'Gemini',
 'Git',
 'Git MongoDB',
 'Git. Java',
 'GitHub',
 'Github',
 'Go',
 'Google O

In [4]:
import pandas as pd
import re
from collections import Counter, defaultdict
from difflib import SequenceMatcher

# =========================
# 1) 데이터 불러오기
# =========================
df = pd.read_excel('./data/DB연동_공고데이터.xlsx')

# =========================
# 2) TECH_STACK 분리
# =========================
raw_tokens = []

for text in df["TECH_STACK"].fillna("").astype(str):
    # 쉼표 기준 분리
    parts = re.split(r",", text)
    for p in parts:
        token = p.strip()
        if token:
            raw_tokens.append(token)

# 빈도수
token_counter = Counter(raw_tokens)

# 원본 표현 목록
raw_unique = sorted(token_counter.keys())

# =========================
# 3) 비교용 키 만들기 함수
# =========================
def make_compare_key(text):
    """
    비교용 키 생성
    - 앞뒤 공백 제거
    - 소문자 변환
    - 특수문자 제거
    - 공백 제거
    """
    text = text.strip().lower()
    text = re.sub(r"[.\-_/()]+", "", text)   # ., -, _, /, () 제거
    text = re.sub(r"\s+", "", text)          # 모든 공백 제거
    return text

# 원본 -> 비교키
key_map = {token: make_compare_key(token) for token in raw_unique}

# =========================
# 4) 비교키가 완전히 같은 것끼리 1차 그룹화
# =========================
exact_groups = defaultdict(list)

for token in raw_unique:
    exact_groups[key_map[token]].append(token)

# exact_groups를 보기 좋게 정리
exact_group_rows = []
for compare_key, variants in exact_groups.items():
    total_count = sum(token_counter[v] for v in variants)
    exact_group_rows.append({
        "compare_key": compare_key,
        "대표후보": sorted(variants, key=lambda x: (-token_counter[x], x))[0],
        "표현들": variants,
        "표현수": len(variants),
        "총빈도": total_count
    })

exact_group_df = pd.DataFrame(exact_group_rows)\
    .sort_values(["표현수", "총빈도"], ascending=[False, False])\
    .reset_index(drop=True)

print("=== 비교키가 완전히 같은 후보 그룹 ===")
print(exact_group_df.head(30))

=== 비교키가 완전히 같은 후보 그룹 ===
    compare_key          대표후보                                         표현들  \
0        jquery        Jquery            [JQUERY, JQuery, Jquery, jQuery]   
1         mysql         MySQL                       [MYSQL, MySQL, MySQl]   
2           git           Git                             [GIT, Git, git]   
3    springboot   Spring Boot       [Spring Boot, SpringBoot, Springboot]   
4    tensorflow    Tensorflow        [TensorFlow, Tensorflow, tensorflow]   
5       reactjs      React.js               [React.JS, React.js, ReactJS]   
6   reactnative  React Native  [React Native, React native, React-Native]   
7    restfulapi   Restful API     [RESTful API, Restful API, Restful APi]   
8          java          JAVA                                [JAVA, Java]   
9    javascript    Javascript                    [JavaScript, Javascript]   
10        react         React                              [React, react]   
11          jsp           JSP                     

In [5]:
# =========================
# 5) 유사 문자열 후보 찾기
# =========================
def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

# exact group의 compare_key 목록
group_keys = list(exact_groups.keys())

similar_rows = []

# 너무 짧은 문자열은 오탐이 많아서 조건을 조금 둠
for i in range(len(group_keys)):
    for j in range(i + 1, len(group_keys)):
        k1 = group_keys[i]
        k2 = group_keys[j]

        # 길이가 너무 짧으면 스킵
        if len(k1) < 3 or len(k2) < 3:
            continue

        score = similarity(k1, k2)

        # 임계값은 상황 따라 조절
        if score >= 0.88:
            vars1 = exact_groups[k1]
            vars2 = exact_groups[k2]

            similar_rows.append({
                "key1": k1,
                "group1": vars1,
                "count1": sum(token_counter[v] for v in vars1),
                "key2": k2,
                "group2": vars2,
                "count2": sum(token_counter[v] for v in vars2),
                "similarity": round(score, 3)
            })

similar_df = pd.DataFrame(similar_rows)\
    .sort_values(["similarity", "count1", "count2"], ascending=[False, False, False])\
    .reset_index(drop=True)

print("=== 유사 문자열 후보 그룹 ===")
print(similar_df.head(50))

=== 유사 문자열 후보 그룹 ===
              key1                                   group1  count1  \
0   spingframework                        [Sping Framework]       1   
1       javascript                 [JavaScript, Javascript]     231   
2       restfulapi  [RESTful API, Restful API, Restful APi]       8   
3       postgresql                 [PostgreSQL, postgresql]       6   
4         websocke                               [WebSocke]       1   
5           jquery         [JQUERY, JQuery, Jquery, jQuery]      56   
6          phython                                [Phython]       1   
7          pythoin                                [Pythoin]       1   
8           jquery         [JQUERY, JQuery, Jquery, jQuery]      56   
9      transformer                            [Transformer]       3   
10      javascript                 [JavaScript, Javascript]     231   
11     전자정부표준프레임워크                            [전자정부표준프레임워크]       7   
12      sprimgboot                [Sprimg Boot, SprimgBo

In [6]:
# =========================
# 6) 엑셀로 저장
# =========================
with pd.ExcelWriter("기술스택_후보그룹_검토용.xlsx", engine="openpyxl") as writer:
    exact_group_df.to_excel(writer, sheet_name="exact_groups", index=False)
    similar_df.to_excel(writer, sheet_name="similar_groups", index=False)

print("저장 완료: 기술스택_후보그룹_검토용.xlsx")

저장 완료: 기술스택_후보그룹_검토용.xlsx


In [7]:
print(freq_df.to_string(index=False))
freq_df.to_excel("기술스택_빈도표.xlsx", index=False)

                     raw_stack  count
                          JAVA    300
                        Python    259
                    Javascript    171
                         React    148
                           CSS    138
                         MySQL    125
                          HTML    119
                           AWS    116
                           Git    113
                           JSP    104
                        Spring    104
                           SQL    101
                         Linux     94
                       Node.js     91
                        Oracle     88
                           API     64
                         Figma     63
                    JavaScript     60
                         HTML5     58
                   Spring Boot     53
                            AI     46
                            C#     36
                           딥러닝     35
                          머신러닝     34
                       Android     33
            

In [9]:
# 기술스택 후보그룹 검토용, 빈도표를 보고 정규화 사전 만들기
normalize_map = {
    'JQUERY' : 'jQuery',
    'JQuery' : 'jQuery',
    'Jquery' : 'jQuery',
    'jQuery' : 'jQuery',
    'jQurery' : 'jQuery',

    'MYSQL' : 'MySQL', 
    'MySQL' : 'MySQL', 
    'MySQl' : 'MySQL',

    'GIT' : 'Git',
    'Git' : 'Git',
    'git' : 'Git',

    'Spring Boot' : 'Spring Boot', 
    'SpringBoot' : 'Spring Boot',
    'Springboot' : 'Spring Boot',
    'Sprimg Boot' : 'Spring Boot', 
    'SprimgBoot' : 'Spring Boot',

    'TensorFlow' : 'TensorFlow', 
    'Tensorflow' : 'TensorFlow',
    'tensorflow' : 'TensorFlow',
    'Tensorflow)활용 경험' : 'TensorFlow',

    # React.js, React를 하나로 모음
    'React.JS' : 'React',
    'React.js' : 'React',
    'ReactJS' : 'React',
    'React' : 'React',
    'react' : 'React',

    'React Native' : 'React Native',
    'React native' : 'React Native',
    'React-Native' : 'React Native',

    # RESTful API, REST API를 하나로 모음
    'RESTful API' : 'REST API', 
    'Restful API' : 'REST API',
    'Restful APi' : 'REST API',
    'Resstful API' : 'REST API',
    'REST API' : 'REST API',
    'RestAPI' : 'REST API',

    'JAVA' : 'Java', 
    'Java' : 'Java',

    'JavaScript' : 'JavaScript',
    'Javascript' : 'JavaScript',
    'x-javascript' : 'JavaScript',
    'JavaScrpit' : 'JavaScript',

    'JSP' : 'JSP',
    'Jsp' : 'JSP',

    'Linux' : 'Linux',
    'Linux.' : 'Linux',
    'Linu' : 'Linux',

    'Node.js' : 'Node.js', 
    'Nodejs' : 'Node.js',

    'Ai' : 'AI',
    'AI' : 'AI',
    '인공지능(AI)' : 'AI',

    'PyTorch' : 'PyTorch',
    'Pytorch' : 'PyTorch',

    'AJAX' : 'Ajax',
    'Ajax' : 'Ajax',

    # CSS3를 CSS로 모음
    'CSS 3' : 'CSS', 
    'CSS3' : 'CSS',

    'Flutter' : 'Flutter', 
    'flutter' : 'Flutter',

    'GitHub' : 'GitHub', 
    'Github' : 'GitHub',

    'PHP' : 'PHP', 
    'php' : 'PHP',

    'BootStrap' : 'Bootstrap', 
    'Bootstrap' : 'Bootstrap',

    'Open CV' : 'OpenCV', 
    'OpenCV' : 'OpenCV',

    'TypeScript' : 'TypeScript', 
    'Typescript' : 'TypeScript',

    'MyBatis' : 'MyBatis',
    'mybatis' : 'MyBatis',

    'IBatis' : 'IBatis',
    'iBATIS' : 'IBatis',

    'Nest.js' : 'NestJS',
    'NestJS' : 'NestJS',

    'PostgreSQL' : 'PostgreSQL', 
    'postgresql' : 'PostgreSQL',
    'Postgrsql' : 'PostgreSQL',

    'Kubernetes' : 'Kubernetes', 
    'kubernetes' : 'Kubernetes',

    'Network' : 'Network', 
    'network' : 'Network',

    'iOS' : 'iOS', 
    'ios' : 'iOS',

    'Hugging Face' : 'Hugging Face',
    'HuggingFace' : 'Hugging Face',

    'Scikit-Learn' : 'Scikit-Learn', 
    'Scikit-learn' : 'Scikit-Learn',

    'TailwindCSS' : 'TailwindCSS', 
    'tailwindcss' : 'TailwindCSS',

    'WebSocket' : 'WebSocket', 
    'Websocket' : 'WebSocket',
    'WebSocke' : 'WebSocket',

    'Spring Framework' : 'Spring Framework',
    'Sping Framework' : 'Spring Framework',

    'Python' : 'Python',
    'Phython' : 'Python',
    'Pythoin' : 'Python',

    '전자정부프레임워크' : '전자정부표준프레임워크',
    '전자정부표준프레임워크' : '전자정부표준프레임워크',

    'Transformwe' : 'Transformer',
    'Transformer' : 'Transformer',

    'HTML5' : 'HTML',
    'HTML' : 'HTML',

    'Word' : '워드',
    '워드' : '워드',

    'C언어' : 'C',
    'C' : 'C',

    '.Net Framrwork' : '.NET Framework',
    'Diango' : 'Django',
    'Excle' : 'Excel',
    'HTLM' : 'HTML',
    'Seienium' : 'Selenium',
    'Spting' : 'Spring',
    'Uneral' : 'Unreal',
    'VScode' : 'VSCode',
    'swagger' : 'Swagger',
    
}